# Análisis On-Chain: Token TWT en BNB Smart Chain

**Wallet:** `0xe2fc31F816A9b94326492132018C3aEcC4a93aE1`  
**Token:** TWT (`0x4B0F1812e5Df2A09796481Ff14017e6005508003`)  
**RPC:** MegaNode (archive node)

Este notebook consulta datos históricos directamente de la blockchain usando `eth_call` y `eth_getLogs`.

## 1. Setup e imports

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
plt.rcParams['figure.dpi'] = 120

from bsc_analyzer import (
    get_balance_at,
    find_transfers,
    sample_balance_trend,
    top_counterparties,
    transfer_summary,
    plot_balance_trend,
    plot_top_counterparties,
)
from bsc_analyzer.config import get_w3, START_BLOCK, WALLET, TWT_CONTRACT

w3 = get_w3()
current_block = w3.eth.block_number
print(f"Conectado a BSC. Bloque actual: {current_block:,}")
print(f"Rango a analizar: {START_BLOCK:,} → {current_block:,}")
print(f"Total bloques: {current_block - START_BLOCK:,}")

## 2. Saldo histórico en bloque 46,080,111

In [ ]:
from decimal import Decimal

balance_inicial = get_balance_at(START_BLOCK)
balance_actual = get_balance_at(current_block)

cambio = balance_actual - balance_inicial
pct_cambio = (cambio / balance_inicial * 100) if balance_inicial > 0 else Decimal(0)

print(f"{'Saldo inicial (bloque {START_BLOCK:,}):':40s} {balance_inicial:>20,.6f} TWT")
print(f"{'Saldo actual (bloque {current_block:,}):':40s} {balance_actual:>20,.6f} TWT")
print(f"{'Cambio neto:':40s} {cambio:>20,.6f} TWT ({pct_cambio:+.2f}%)")

## 3. Tendencia de saldo (cada 1,000,000 bloques)

Muestrea el balance cada millón de bloques para visualizar la evolución.

In [ ]:
trend_data = sample_balance_trend(step=1_000_000)
print(f"Muestras obtenidas: {len(trend_data)}")

In [ ]:
plot_balance_trend(trend_data, title=f"TWT Balance — Wallet {WALLET[:10]}...")

## 4. Tracking de transferencias BEP-20

Escanea inteligentemente los bloques con actividad usando detección de cambios de balance.
⚠️ Este paso puede tomar 1-3 minutos dependiendo del número de transacciones.

In [ ]:
transfers = find_transfers()
print(f"Total transferencias encontradas: {len(transfers)}")

In [ ]:
# Resumen de transferencias
summary = transfer_summary(transfers)
print("=" * 50)
print(" RESUMEN DE TRANSFERENCIAS")
print("=" * 50)
print(f"  Total transacciones:      {summary['total_txs']:>8,}")
print(f"  Entrantes:                {summary['incoming_count']:>8,}  (+{summary['total_in_twt']:,.2f} TWT)")
print(f"  Salientes:                {summary['outgoing_count']:>8,}  (-{summary['total_out_twt']:,.2f} TWT)")
print(f"  Flujo neto:               {summary['net_flow_twt']:>15,.2f} TWT")
print(f"  Contrapartes únicas:      {summary['unique_counterparties']:>8,}")

### Últimas 20 transferencias

In [ ]:
import pandas as pd

rows = []
for t in transfers[-20:]:
    contraparte = t.sender if t.direction == 'IN' else t.receiver
    rows.append({
        'Bloque': t.block_number,
        'Dir.': t.direction,
        'Contraparte': f"{contraparte[:8]}...{contraparte[-6:]}",
        'Monto (TWT)': f"{t.amount:,.2f}",
        'TxHash': f"{t.tx_hash[:10]}..."
    })

df = pd.DataFrame(rows)
df.style.set_caption("Últimas 20 transferencias de TWT")

## 5. Top 10 contrapartes (transferencias salientes)

In [ ]:
top10 = top_counterparties(transfers, top_n=10)

print(f"{'Dirección':42s} {'Total TWT':>18s} {'# Txs':>6s}")
print("-" * 68)
for addr, amt, cnt in top10:
    print(f"{addr:42s} {amt:>18,.2f} {cnt:>6,}")

In [ ]:
plot_top_counterparties(top10, title="Top 10 — Destinos de TWT enviados")

## 6. Consulta puntual: saldo en cualquier bloque

Cambia el valor de `bloque` para consultar un bloque específico.

In [ ]:
bloque = 46080111  # ← Cambia este valor
saldo = get_balance_at(bloque)
print(f"Saldo TWT en bloque {bloque:,}: {saldo:,.6f} TWT")